# Daily Challenge: RAG with FAISS and ChromaDB

## Exercise 1: Data Loading and Preparation

### 1. Install Required Libraries

In [55]:
!pip install -q faiss-cpu==1.14.3

In [56]:
!pip install -q chromadb==0.3.21

In [57]:
!pip install -qU chromadb

In [58]:
!pip install -q "numpy<2"

In [59]:
# Create a Cache Directory and install libomp-dev
!mkdir -p cache
!apt install -y libomp-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libomp-dev is already the newest version (1:14.0-55~exp2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [60]:
!python -m pip install --upgrade faiss-cpu

#### Import essential libraries

In [61]:
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [62]:
# Download the dataset
!wget -q https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/labelled_newscatcher_dataset.csv -P /content/

### 2. Load the Dataset

In [66]:
import os
import pandas as pd
import requests

# Assuming the dataset is in the content directory
path = '/content/labelled_newscatcher_dataset.csv'
# Corrected URL for the dataset
url = 'https://raw.githubusercontent.com/pinecone-io/examples/master/docs/gen/ai-for-everyone/newscatcher-rag/labelled_newscatcher_dataset.csv'

# Check if the file exists, and if not, download it using requests
if not os.path.exists(path):
    print(f"Dataset not found at {path}, attempting to download from {url}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Dataset downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Failed to download dataset: {e}")
        # Optionally, raise an error or handle further if download definitely failed
        raise FileNotFoundError(f"Dataset not found and failed to download: {path}")

try:
    pdf = pd.read_csv(path)
    print(f"Dataset loaded with {len(pdf)} rows.")
except FileNotFoundError:
    print(f"Error: The file {path} was not found after download attempt. Please check the path and download process.")
    raise
except Exception as e:
    print(f"An unexpected error occurred while reading the CSV: {e}")
    raise

Dataset not found at /content/labelled_newscatcher_dataset.csv, attempting to download from https://raw.githubusercontent.com/pinecone-io/examples/master/docs/gen/ai-for-everyone/newscatcher-rag/labelled_newscatcher_dataset.csv...
Failed to download dataset: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/pinecone-io/examples/master/docs/gen/ai-for-everyone/newscatcher-rag/labelled_newscatcher_dataset.csv


FileNotFoundError: Dataset not found and failed to download: /content/labelled_newscatcher_dataset.csv

### 3. Add an Identifier Column (if needed)

In [ ]:
pdf["id"] = np.arange(len(pdf))

### 4. Inspect the Data

In [ ]:
display(pdf.head())

### 5. Create a Subset for Faster Processing

In [ ]:
# Select a smaller subset (e.g., the first 1000 rows)
pdf_subset = pdf.head(1000).copy()
print(f"Created subset with {len(pdf_subset)} rows.")
display(pdf_subset.head())

## Exercise 2: Vectorization with Sentence Transformers

### 1. Install and Import Sentence Transformers Library

In [ ]:
# InputExample was already imported in the initial imports cell (f805cb61)

### 2. Prepare the Data for Embedding Generation

In [ ]:
# pdf_subset was already created in Exercise 1.
display(pdf_subset.head())

### 3. Create a Helper Function

In [ ]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    # Assuming the 'title' column contains the text for embedding
    return InputExample(texts=[doc1['title']])

### 4. Apply the Helper Function to the Subset

In [ ]:
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
print(faiss_train_examples[:10])

### 5. Initialize the Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'  # Specify the model name.
)
print("Embedding model 'all-MiniLM-L6-v2' initialized.")

### 6. Extract the Titles and Convert to a List of Strings

In [ ]:
titles_list = pdf_subset["title"].tolist()
print(f"Extracted {len(titles_list)} titles.")
print(titles_list[:5])

### 7. Generate Embeddings for the Titles

In [ ]:
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)
print("Embeddings generated.")

### 8. Check Embedding Dimensions

In [ ]:
print(f"Number of embeddings: {len(faiss_title_embedding)}, Dimension of each embedding: {len(faiss_title_embedding[0])}")

## Exercise 3: FAISS Indexing and Search

### 1. Install and Import FAISS Library

In [ ]:
# numpy and faiss were already imported in the initial imports cell (f805cb61)

### 2. Prepare the Data for Indexing

In [ ]:
pdf_to_index = pdf_subset # This should be your subset DataFrame containing the articles.
id_index = pdf_subset["id"].values # An array of IDs corresponding to each article.
print(f"Prepared {len(pdf_to_index)} articles for indexing.")

### 3. Normalize the Embedding Vectors

In [ ]:
content_encoded_normalized = faiss_title_embedding.astype('float32') # Embedding vectors.
faiss.normalize_L2(content_encoded_normalized)
print("Embedding vectors normalized.")

### 4. Create the FAISS Index

In [ ]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f"FAISS index created with {index_content.ntotal} vectors.")

### 5. Implement a Search Function

In [ ]:
def search_content(query_text, pdf_to_index, k=3):
    query_vector = model.encode([query_text]).astype('float32') # Encode the query string into an embedding vector.
    faiss.normalize_L2(query_vector)  # Normalize the query vector.

    # Perform the search
    distances, ids = index_content.search(query_vector, k) # The top-k similar vectors.
    ids = ids[0].tolist()
    similarities = distances[0].tolist()

    results = pdf_to_index[pdf_to_index['id'].isin(ids)].copy() # Retrieve the matching articles from pdf_to_index.
    results['faiss_id'] = results['id'].apply(lambda x: ids.index(x)) # Add a column to maintain order based on FAISS results
    results = results.sort_values(by='faiss_id').drop(columns=['faiss_id'])
    results["similarities"] = similarities  # Add similarity scores.
    return results

### 6. Test the Search Function

In [ ]:
display(search_content("animal", pdf_to_index, k=5))

## Exercise 4: ChromaDB Collection and Querying

### 1. Install and Import ChromaDB Library

In [ ]:
# chromadb and Settings were already imported in the initial imports cell (f805cb61)

### 2. Initialize a ChromaDB Client and Create a Collection

In [ ]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts
if collection_name in [col.name for col in chroma_client.list_collections()]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

### 3. Add Data to the Collection

In [ ]:
# Display the DataFrame subset (for reference)
display(pdf_subset.head())

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]  # Provide a list of unique IDs.
)
print(f"Added {collection.count()} documents to the ChromaDB collection.")

### 4. Query the Collection

In [ ]:
import json

results = collection.query(
    query_texts=["space"],  # Perform the search query.
    n_results=10,
    include=['documents', 'metadatas', 'distances']
)

print(json.dumps(results, indent=4))

## Exercise 5: Question Answering with Hugging Face Model

### 1. Install and Import the Transformers Library

In [ ]:
# AutoTokenizer, AutoModelForCausalLM, pipeline were already imported in the initial imports cell (f805cb61)

### 2. Initialize the Model and Tokenizer

In [ ]:
model_id = 'gpt2'  # Specify the Hugging Face model ID (e.g., 'gpt2').
tokenizer = AutoTokenizer.from_pretrained(model_id)  # Load the tokenizer for the model.
lm_model = AutoModelForCausalLM.from_pretrained(model_id)  # Load the causal language model.

# Ensure the tokenizer has a pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    lm_model.config.pad_token_id = lm_model.config.eos_token_id

print(f"Model '{model_id}' and tokenizer initialized.")

### 3. Create a Text Generation Pipeline

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,  # Maximum number of tokens to generate.
    device_map="auto",   # Automatically uses available GPU/CPU resources.
)
print("Text generation pipeline created.")

### 4. Construct a Prompt Template

In [ ]:
# Ensure 'question' and 'results' are defined from previous steps (Exercise 4, Query the Collection)
# If running this cell independently, ensure 'results' is available from a ChromaDB query

# Example values if running standalone (replace with actual if needed)
if 'question' not in locals():
    question = "What are some recent discoveries about space exploration?"
    print(f"Using default question: {question}")

if 'results' not in locals() or not results['documents']:
    print("Warning: 'results' from ChromaDB query not found or empty. Using placeholder context.")
    # Fallback context if ChromaDB query wasn't run or returned nothing
    context = "The latest news in space exploration includes new findings from the James Webb Space Telescope and plans for future Mars missions."
else:
    context_docs = results["documents"][0] # Assuming results['documents'] is a list of lists of strings
    context = " ".join([f"#{doc}" for doc in context_docs])  # Concatenate the retrieved documents.
    print("Using context from ChromaDB query results.")

prompt_template = f"Relevant context: {context}\n\nThe user's question: {question}\n\nAnswer:"
print("Prompt template constructed:")
print(prompt_template)

### 5. Generate a Response Using the Pipeline

In [ ]:
lm_response = pipe(prompt_template)  # Use the pipeline to generate text based on the prompt.
print("\nGenerated Response:")
print(lm_response[0]["generated_text"])

### 6. Experiment with Different Prompts and Context Windows

Feel free to modify the `question` variable and the `n_results` parameter in the ChromaDB query to observe how the language model's responses change with different questions and varying amounts of retrieved context. You can re-run the relevant cells to test new prompts.

## Part 1: Load Documents & Execute Reranking Model

### 1. Install Pinecone libraries

In [ ]:
pip install -U pinecone==6.0.1 pinecone-notebooks

### 2. Authenticate with Pinecone

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

### 3. Instantiate the Pinecone client

In [ ]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

### 4. Define your query & documents

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "Apples are a type of fruit that grow on trees. They are crisp and can be red, green, or yellow.", # Add a document about apple fruit
    "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide.", # Add a document about Apple company products
    "An apple a day keeps the doctor away is a common English proverb, suggesting that eating apples is good for health.", # Add another fruit-related document
    "The iPhone is a line of smartphones designed and marketed by Apple Inc. These devices use Apple's iOS mobile operating system.", # Add another company-related document
    "Many recipes call for apples, such as apple pie, apple sauce, and apple cider." # Add one more document (your choice)
]

### 5. Call the reranker

In [ ]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # e.g., 3
)

### 6. Inspect reranked results

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f}, Document: {m.document.text}")

show_reranked_results(query, reranked.matches)

## Part 2: Setup a Serverless Index for Medical Notes

### 1. Install data & model libraries

In [ ]:
!pip install pandas torch transformers

### 2. Import modules & define environment settings

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index' # Give your index a name

### 3. Create or recreate the index

In [ ]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384, # This matches our embedding model size
    metric='cosine', # Distance metric for similarity
    spec=spec
)

## Part 3: Load the Sample Data

### 1. Download & read JSONL

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

### 2. Preview the DataFrame

In [ ]:
# Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
df.head()

## Part 4: Upsert Data into the Index

### 1. Instantiate index client & upsert

In [ ]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)

### 2. Wait for availability

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0 # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

## Part 5: Query & Embedding Function

### 1. Define your embedding function

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=1) # Which dimension to average?
    return embedding

### 2. Run a semantic search query

In [ ]:
# Build a query to search
question = "What are common symptoms and treatments for acute bronchitis?" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Part 6: Display & Rerank Clinical Notes

### 1. Display initial search results

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]:.4f}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)

### 2. Prepare documents for reranking

In [ ]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

### 3. Execute serverless reranking

In [ ]:
# Define a more specific query for reranking
refined_query = "Treatment options for acute bronchitis with chest pain" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)

### 4. Show reranked results

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score:.4f}') # What attribute contains the reranking score?
        print(f' Reranking Field: {match.document.reranking_field}') # What contains the searchable field?
        print('')
show_reranked_results(refined_query, reranked_results.matches)

### 5. Clean up (optional)

In [ ]:
# Delete the index to save resources
pc.delete_index(name=index_name)